# Notebook 02 — NASA POWER Data Downloader

This notebook queries the NASA POWER API to retrieve monthly meteorological data (2000–2025) for each Sri Lankan coconut-growing district, saves individual district CSV files, and combines them into a master weather dataset.

## 1. Import Packages & Settings

In [13]:
import sys
import os
import time
import requests
import pandas as pd
import numpy as np
from pathlib import Path

# Ensure project root is in the path
sys.path.append(str(Path(os.getcwd()).parent))

import config
print("Raw Data Folder:", config.RAW_DIR)

Raw Data Folder: D:\A1\coco\Coconut_Yield_Prediction\raw


## 2. Define the NASA POWER API Query Function

We will query monthly data. The variables requested are:
- `PRECTOTCORR` (Precipitation corrected - Rainfall in mm/day)
- `T2M` (Mean Temperature at 2m in °C)
- `T2M_MAX` (Max Temperature at 2m in °C)
- `T2M_MIN` (Min Temperature at 2m in °C)
- `RH2M` (Relative Humidity at 2m in %)
- `ALLSKY_SFC_SW_DWN` (Solar Radiation in MJ/m²/day)
- `WS2M` (Wind Speed at 2m in m/s)

Note on Evapotranspiration (ET): Since NASA POWER monthly API does not always have a direct `EVAP` parameter in the Agrometeorology community, we will query `EVPTRNS` (Potential Evapotranspiration) and calculate potential evapotranspiration (PET) using the Hargreaves-Samani method as a robust backup.

In [14]:
def calculate_hargreaves_pet(row):
    """
    Hargreaves-Samani equation for Potential Evapotranspiration (PET):
    PET = 0.0023 * RA * (Tmean + 17.8) * (Tmax - Tmin)**0.5
    RA (Extraterrestrial radiation) can be approximated from All-Sky Solar Radiation.
    """
    t_mean = row['T2M']
    t_max = row['T2M_MAX']
    t_min = row['T2M_MIN']
    solar_rad = row['ALLSKY_SFC_SW_DWN']
    
    # Standard Hargreaves-Samani temperature difference
    t_range = max(t_max - t_min, 0.1)
    
    # Extraterrestrial radiation approximation (MJ/m2/day) from solar rad at ground
    # Typically RA is larger than ground solar radiation due to atmosphere absorption. 
    # Atmospheric transmissivity is roughly 0.5 - 0.7. We'll use 0.65 as an average factor.
    ra_approx = solar_rad / 0.65
    
    # Evapotranspiration in mm/day
    pet_mm_day = 0.0023 * ra_approx * (t_mean + 17.8) * (t_range ** 0.5)
    return max(pet_mm_day, 0.0)

def download_district_weather(district_name, lat, lon, start_year=2000, end_year=2025):
    params = {
        "latitude": lat,
        "longitude": lon,
        "start": str(start_year),
        "end": str(end_year),
        "parameters": "PRECTOTCORR,T2M,T2M_MAX,T2M_MIN,RH2M,ALLSKY_SFC_SW_DWN,WS2M,EVPTRNS",
        "community": "AG",
        "format": "JSON"
    }
    
    retries = 3
    for attempt in range(retries):
        try:
            print(f"Downloading {district_name} (Attempt {attempt+1}/{retries})...")
            response = requests.get(config.NASA_POWER_URL, params=params, timeout=30)
            if response.status_code == 200:
                data = response.json()
                parameter_data = data["properties"]["parameter"]
                
                # Reformat dict structure into DataFrame
                df_dict = {}
                for param, monthly_values in parameter_data.items():
                    df_dict[param] = monthly_values
                
                df = pd.DataFrame(df_dict)
                df.index.name = "YYYYMM"
                df = df.reset_index()
                
                # Split YYYYMM index into Year and Month columns
                df["Year"] = df["YYYYMM"].apply(lambda x: int(str(x)[:4]))
                df["Month"] = df["YYYYMM"].apply(lambda x: int(str(x)[4:]))
                df["District"] = district_name
                
                # Normalize Rainfall from mm/day to total mm in month
                # (assuming average month length of 30.4 days or dynamic based on month)
                days_in_month = df["Month"].map({1:31, 2:28, 3:31, 4:30, 5:31, 6:30, 7:31, 8:31, 9:30, 10:31, 11:30, 12:31})
                # Leap year adjustment for February
                is_leap = (df["Year"] % 4 == 0) & ((df["Year"] % 100 != 0) | (df["Year"] % 400 == 0))
                days_in_month = np.where((df["Month"] == 2) & is_leap, 29, days_in_month)
                
                df["Rainfall"] = df["PRECTOTCORR"] * days_in_month
                df["Temperature"] = df["T2M"]
                df["Humidity"] = df["RH2M"]
                df["Solar_Radiation"] = df["ALLSKY_SFC_SW_DWN"]
                df["Wind_Speed"] = df["WS2M"]
                
                # Handle Evapotranspiration
                # If EVPTRNS (potential evapotranspiration) is available in API and not all -999, use it.
                # Else, calculate PET using Hargreaves-Samani.
                if "EVPTRNS" in df.columns and not (df["EVPTRNS"] == -999).all():
                    df["Evapotranspiration"] = df["EVPTRNS"] * days_in_month
                else:
                    # Calculate Hargreaves PET
                    df["Evapotranspiration"] = df.apply(calculate_hargreaves_pet, axis=1) * days_in_month
                
                # Select and order final columns
                cols = ["District", "Year", "Month", "Rainfall", "Temperature", "Humidity", "Solar_Radiation", "Wind_Speed", "Evapotranspiration"]
                df = df[cols]
                
                return df
            else:
                print(f"  [WARN] Request failed with status code {response.status_code}")
        except Exception as e:
            print(f"  [WARN] Exception occurred: {str(e)}")
        time.sleep(2)  # Delay between retries
    
    raise RuntimeError(f"Failed to download weather data for {district_name} after {retries} attempts.")

## 3. Execute Download Loop for All Districts

In [15]:
district_dfs = []

for name, coords in config.DISTRICTS.items():
    try:
        df = download_district_weather(name, coords["lat"], coords["lon"], config.START_YEAR, config.END_YEAR)
        district_dfs.append(df)
        # Save individual district file
        out_path = config.RAW_DIR / f"weather_{name.lower()}.csv"
        df.to_csv(out_path, index=False)
        print(f"  [SAVED] {out_path.name} (Rows: {len(df)})")
        time.sleep(1) # API rate limit avoidance
    except Exception as e:
        print(f"[ERROR] Skipping {name}: {str(e)}")

print(f"Successfully downloaded data for {len(district_dfs)}/{len(config.DISTRICTS)} districts.")

  [WARN] Exception occurred: HTTPSConnectionPool(host='power.larc.nasa.gov', port=443): Read timed out. (read timeout=30)
  [SAVED] weather_kurunegala.csv (Rows: 338)
  [WARN] Exception occurred: HTTPSConnectionPool(host='power.larc.nasa.gov', port=443): Read timed out.
  [SAVED] weather_puttalam.csv (Rows: 338)
  [SAVED] weather_gampaha.csv (Rows: 338)
  [SAVED] weather_kalutara.csv (Rows: 338)
  [SAVED] weather_colombo.csv (Rows: 338)
  [SAVED] weather_galle.csv (Rows: 338)
  [SAVED] weather_matara.csv (Rows: 338)
  [SAVED] weather_hambantota.csv (Rows: 338)
  [SAVED] weather_kegalle.csv (Rows: 338)
  [SAVED] weather_ratnapura.csv (Rows: 338)
  [SAVED] weather_ampara.csv (Rows: 338)
  [SAVED] weather_batticaloa.csv (Rows: 338)
  [SAVED] weather_jaffna.csv (Rows: 338)
Successfully downloaded data for 13/13 districts.


## 4. Merge All Districts into One Weather Dataset

In [16]:
if district_dfs:
    merged_df = pd.concat(district_dfs, ignore_index=True)
    master_weather_path = config.RAW_DIR / "weather_raw.csv"
    merged_df.to_csv(master_weather_path, index=False)
    print(f"[SUCCESS] Merged and saved weather data to {master_weather_path.name}")
    print(merged_df.info())
    print(merged_df.head())
else:
    print("[ERROR] No weather data was downloaded to merge.")

[SUCCESS] Merged and saved weather data to weather_raw.csv
<class 'pandas.DataFrame'>
RangeIndex: 4394 entries, 0 to 4393
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   District            4394 non-null   str    
 1   Year                4394 non-null   int64  
 2   Month               4394 non-null   int64  
 3   Rainfall            4056 non-null   float64
 4   Temperature         4394 non-null   float64
 5   Humidity            4394 non-null   float64
 6   Solar_Radiation     4394 non-null   float64
 7   Wind_Speed          4394 non-null   float64
 8   Evapotranspiration  4056 non-null   float64
dtypes: float64(6), int64(2), str(1)
memory usage: 309.1 KB
None
     District  Year  Month  Rainfall  Temperature  Humidity  Solar_Radiation  \
0  Kurunegala  2000      1     25.73        23.43     82.96            16.51   
1  Kurunegala  2000      2     67.57        25.43     75.89            19.25   
2 